# 茶园植物健康环境监测数据分析

**数据来源**：Kaggle公开数据集 plant_health_data.csv（真实生物传感器采集数据）

**核心分析问题**：
1. 各环境指标（温度、湿度、土壤湿度、光照）全期如何变化？
2. 不同监测植株之间存在哪些环境差异？
3. 光照强度与叶绿素含量之间是否存在相关关系？
4. 不同健康状态下的环境与营养指标有何特征？


## 1. 数据读取与基本探索

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

df = pd.read_csv('../data/plant_health_data.csv')
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df['date'] = df['Timestamp'].dt.date
df['hour'] = df['Timestamp'].dt.hour
df['day'] = df['Timestamp'].dt.day
df['Plant_ID_str'] = df['Plant_ID'].apply(lambda x: f'植株{int(x):02d}')

print("数据基本信息：")
print(f"记录数量：{len(df)}")
print(f"时间范围：{df['Timestamp'].min()} 至 {df['Timestamp'].max()}")
print(f"监测植株数：{df['Plant_ID'].nunique()} 株")
print(f"健康状态分类：{df['Plant_Health_Status'].unique()}")
print()
print("字段说明：")
field_desc = {
    'Timestamp': '采集时间',
    'Plant_ID': '植株编号',
    'Soil_Moisture': '土壤湿度 (%)',
    'Ambient_Temperature': '环境温度 (℃)',
    'Soil_Temperature': '土壤温度 (℃)',
    'Humidity': '空气湿度 (%)',
    'Light_Intensity': '光照强度 (lux)',
    'Soil_pH': '土壤pH值',
    'Nitrogen_Level': '氮含量 (mg/kg)',
    'Phosphorus_Level': '磷含量 (mg/kg)',
    'Potassium_Level': '钾含量 (mg/kg)',
    'Chlorophyll_Content': '叶绿素含量 (SPAD)',
    'Plant_Health_Status': '植物健康状态',
}
for k,v in field_desc.items():
    print(f"  {k}: {v}")
print()
print(df.describe().round(2))


## 2. 数据清洗与处理

In [ ]:
print(" 缺失值检查")
print(df.isnull().sum())

print("\n异常值检测（IQR方法")
num_cols = ['Soil_Moisture','Ambient_Temperature','Soil_Temperature','Humidity','Light_Intensity']
for col in num_cols:
    q1,q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3-q1
    outliers = df[(df[col]<q1-1.5*iqr)|(df[col]>q3+1.5*iqr)]
    print(f"{col} 异常值：{len(outliers)} 条")

# 清洗
df_clean = df.copy()
for col in num_cols:
    q1,q3 = df_clean[col].quantile(0.25), df_clean[col].quantile(0.75)
    iqr = q3-q1
    df_clean[col] = df_clean[col].clip(q1-1.5*iqr, q3+1.5*iqr)

# 衍生指标
df_clean['Temp_Diff'] = df_clean['Ambient_Temperature'] - df_clean['Soil_Temperature']
df_clean['Nutrient_Index'] = (df_clean['Nitrogen_Level'] + df_clean['Phosphorus_Level'] + df_clean['Potassium_Level']) / 3

# 中文健康状态映射（后续图表统一使用中文标签）
df_clean['健康状态'] = df_clean['Plant_Health_Status'].map({
    'Healthy': '健康',
    'Moderate Stress': '中度胁迫',
    'High Stress': '高度胁迫'
})

print("\n健康状态分布：")
print(df_clean['健康状态'].value_counts())
print("\n衍生指标：")
print("Temp_Diff: 气土温差 = 环境温度 - 土壤温度")
print("Nutrient_Index: 综合营养指数 = (N + P + K) / 3")

In [ ]:
# ===== 数据概览面板 =====
print("=" * 60)
print("  茶园植物健康环境监测 — 数据概览")
print("=" * 60)
print(f"  记录总数：{len(df_clean):,} 条")
print(f"  监测植株：{df_clean['Plant_ID'].nunique()} 株")
print(f"  时间跨度：{df_clean['Timestamp'].min().strftime('%Y-%m-%d')} ~ {df_clean['Timestamp'].max().strftime('%Y-%m-%d')}")
print(f"  监测天数：{df_clean['day'].nunique()} 天")
print(f"  环境指标：气温 / 土温 / 空气湿度 / 土壤湿度 / 光照 / 土壤pH")
print(f"  营养指标：氮 / 磷 / 钾 / 叶绿素")
print(f"  电化学信号：有")
print("-" * 60)

# 健康状态分布饼图
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

health_counts = df_clean['健康状态'].value_counts()
health_colors = {'健康': '#59A96C', '中度胁迫': '#E5A93D', '高度胁迫': '#E0555A'}
colors_pie = [health_colors[k] for k in health_counts.index]
wedges, texts, autotexts = ax1.pie(
    health_counts.values, labels=health_counts.index, autopct='%1.1f%%',
    colors=colors_pie, explode=(0.03, 0.03, 0.03),
    textprops={'fontsize': 12}, pctdistance=0.6
)
for at in autotexts:
    at.set_fontweight('bold')
    at.set_fontsize(11)
ax1.set_title('植物健康状态分布', fontsize=14, fontweight='bold')

# 关键指标均值对比（健康 vs 高度胁迫）
healthy = df_clean[df_clean['健康状态'] == '健康']
stressed = df_clean[df_clean['健康状态'] == '高度胁迫']
key_metrics = ['Soil_Moisture', 'Nitrogen_Level', 'Ambient_Temperature', 'Humidity', 'Chlorophyll_Content']
key_labels = ['土壤湿度\n(%)', '氮含量\n(mg/kg)', '环境温度\n(℃)', '空气湿度\n(%)', '叶绿素\n(SPAD)']
x = np.arange(len(key_labels))
w = 0.35
ax2.bar(x - w/2, [healthy[m].mean() for m in key_metrics], w, color='#59A96C', label='健康', edgecolor='white')
ax2.bar(x + w/2, [stressed[m].mean() for m in key_metrics], w, color='#E0555A', label='高度胁迫', edgecolor='white')
ax2.set_xticks(x); ax2.set_xticklabels(key_labels, fontsize=9)
ax2.set_ylabel('均值', fontsize=11)
ax2.set_title('健康 vs 高度胁迫 关键指标对比', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/fig00_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("概览面板 保存完成")

## 3. 可视化分析

### 图1：各指标日均变化趋势（折线图）

In [ ]:

daily = df_clean.groupby('date')[['Ambient_Temperature','Humidity','Soil_Moisture','Light_Intensity']].mean()
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('茶园植物健康环境监测 - 各指标日均变化趋势', fontsize=15, fontweight='bold')

configs = [
    ('Ambient_Temperature', '环境温度 (℃)', axes[0,0], '#E74C3C'),
    ('Humidity', '空气湿度 (%)', axes[0,1], '#3498DB'),
    ('Soil_Moisture', '土壤湿度 (%)', axes[1,0], '#27AE60'),
    ('Light_Intensity', '光照强度 (lux)', axes[1,1], '#F39C12'),
]
for col, ylabel, ax, color in configs:
    ax.plot(range(len(daily)), daily[col], color=color, linewidth=2, marker='o', markersize=4)
    ax.fill_between(range(len(daily)), daily[col], alpha=0.15, color=color)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_xlabel('监测天数', fontsize=10)
    ax.set_title(ylabel.split(' ')[0] + '日均变化', fontsize=12)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/fig01_daily_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print("图1保存完成")


### 图2：各监测植株环境指标对比（条形图）

In [ ]:

plant_avg = df_clean.groupby('Plant_ID_str')[['Ambient_Temperature','Humidity','Soil_Moisture']].mean()
colors10 = ['#E74C3C','#3498DB','#27AE60','#F39C12','#9B59B6','#1ABC9C','#E67E22','#2ECC71','#E91E63','#00BCD4']

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle('各监测植株环境指标对比', fontsize=15, fontweight='bold')

for i, (col, ylabel) in enumerate([
    ('Ambient_Temperature', '环境温度 (℃)'),
    ('Humidity', '空气湿度 (%)'),
    ('Soil_Moisture', '土壤湿度 (%)'),
]):
    bars = axes[i].bar(range(len(plant_avg)), plant_avg[col], color=colors10, edgecolor='white')
    axes[i].set_xticks(range(len(plant_avg)))
    axes[i].set_xticklabels(plant_avg.index, rotation=45, fontsize=8)
    axes[i].set_title(ylabel.split(' ')[0] + '对比', fontsize=12)
    axes[i].set_ylabel(ylabel, fontsize=10)
    axes[i].grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, plant_avg[col]):
        axes[i].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                     f'{val:.1f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('../outputs/figures/fig02_plant_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("图2保存完成")


### 图3：不同健康状态下温度与土壤湿度分布（箱线图）

In [ ]:
status_palette = {'健康': '#59A96C', '中度胁迫': '#E5A93D', '高度胁迫': '#E0555A'}
order = ['健康', '中度胁迫', '高度胁迫']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('不同健康状态下温度与土壤湿度分布', fontsize=14, fontweight='bold')

sns.boxplot(data=df_clean, x='健康状态', y='Ambient_Temperature',
            order=order, palette=status_palette, ax=axes[0], width=0.5)
axes[0].set_title('不同健康状态下环境温度分布', fontsize=12)
axes[0].set_xlabel('健康状态'); axes[0].set_ylabel('环境温度 (℃)'); axes[0].grid(axis='y', alpha=0.3)

sns.boxplot(data=df_clean, x='健康状态', y='Soil_Moisture',
            order=order, palette=status_palette, ax=axes[1], width=0.5)
axes[1].set_title('不同健康状态下土壤湿度分布', fontsize=12)
axes[1].set_xlabel('健康状态'); axes[1].set_ylabel('土壤湿度 (%)'); axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/fig03_health_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()
print("图3保存完成")

### 图4：光照强度与叶绿素含量相关性（散点图）

In [ ]:
status_palette = {'健康': '#59A96C', '中度胁迫': '#E5A93D', '高度胁迫': '#E0555A'}
fig, ax = plt.subplots(figsize=(10, 7))

for status, group in df_clean.groupby('健康状态'):
    ax.scatter(group['Light_Intensity'], group['Chlorophyll_Content'],
               c=status_palette.get(status, 'gray'), label=status, alpha=0.5, s=25, edgecolors='none')

z = np.polyfit(df_clean['Light_Intensity'], df_clean['Chlorophyll_Content'], 1)
p = np.poly1d(z)
xl = np.linspace(df_clean['Light_Intensity'].min(), df_clean['Light_Intensity'].max(), 100)
ax.plot(xl, p(xl), 'k--', linewidth=2, label='趋势线')

corr = df_clean['Light_Intensity'].corr(df_clean['Chlorophyll_Content'])
ax.set_title(f'光照强度与叶绿素含量相关性分析 (r={corr:.3f})', fontsize=13, fontweight='bold')
ax.set_xlabel('光照强度 (lux)'); ax.set_ylabel('叶绿素含量 (SPAD)')
ax.legend(title='健康状态'); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/fig04_light_chlorophyll.png', dpi=150, bbox_inches='tight')
plt.show()
print("图4保存完成")

### 图5：各植株逐日温度与湿度热力图

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('各植株逐日温度与湿度热力图', fontsize=14, fontweight='bold')

pt = df_clean.groupby(['Plant_ID_str', 'day'])['Ambient_Temperature'].mean().unstack()
ph = df_clean.groupby(['Plant_ID_str', 'day'])['Humidity'].mean().unstack()

sns.heatmap(pt, cmap='RdYlGn_r', ax=axes[0], linewidths=0.3,
            cbar_kws={'label': '温度 (℃)'}, annot=False)
axes[0].set_title('各植株逐日环境温度 (℃)', fontsize=12)
axes[0].set_xlabel('日期（天）'); axes[0].set_ylabel('植株编号')

sns.heatmap(ph, cmap='Blues', ax=axes[1], linewidths=0.3,
            cbar_kws={'label': '湿度 (%)'}, annot=False)
axes[1].set_title('各植株逐日空气湿度 (%)', fontsize=12)
axes[1].set_xlabel('日期（天）'); axes[1].set_ylabel('植株编号')

plt.tight_layout()
plt.savefig('../outputs/figures/fig05_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("图5保存完成")


### 图6：各指标相关性矩阵（热力图）

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

corr_cols = ['Ambient_Temperature','Soil_Temperature','Humidity','Soil_Moisture',
             'Light_Intensity','Soil_pH','Nitrogen_Level','Chlorophyll_Content']
corr_labels = ['环境温度','土壤温度','空气湿度','土壤湿度','光照强度','土壤pH','氮含量','叶绿素']

cm = df_clean[corr_cols].corr()
cm.index = corr_labels; cm.columns = corr_labels

sns.heatmap(cm, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.5, square=True, vmin=-1, vmax=1,
            cbar_kws={'label': '相关系数', 'shrink': 0.8})
ax.set_title('各环境与生理指标相关性矩阵', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/figures/fig06_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print("图6保存完成")

### 图7：不同健康状态氮含量与叶绿素分布（直方图）

In [ ]:
status_palette = {'健康': '#59A96C', '中度胁迫': '#E5A93D', '高度胁迫': '#E0555A'}
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('不同健康状态下氮含量与叶绿素含量分布', fontsize=14, fontweight='bold')

for status, color in status_palette.items():
    subset = df_clean[df_clean['健康状态'] == status]
    axes[0].hist(subset['Nitrogen_Level'], bins=20, alpha=0.6, color=color, label=status, edgecolor='white')
    axes[1].hist(subset['Chlorophyll_Content'], bins=20, alpha=0.6, color=color, label=status, edgecolor='white')

axes[0].set_title('氮含量分布（按健康状态）', fontsize=12)
axes[0].set_xlabel('氮含量 (mg/kg)'); axes[0].set_ylabel('频次')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].set_title('叶绿素含量分布（按健康状态）', fontsize=12)
axes[1].set_xlabel('叶绿素含量 (SPAD)'); axes[1].set_ylabel('频次')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/fig07_health_hist.png', dpi=150, bbox_inches='tight')
plt.show()
print("图7保存完成")

### 图8：温度与湿度逐日变化趋势（上下分开展示）

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

daily = df_clean.groupby('date')[['Ambient_Temperature','Humidity']].mean().reset_index()
daily['date_str'] = daily['date'].astype(str)

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=('环境温度逐日变化', '空气湿度逐日变化'),
    row_heights=[1, 1]
)

# 上图：温度
fig.add_trace(
    go.Scatter(
        x=daily['date_str'], y=daily['Ambient_Temperature'],
        mode='lines+markers',
        name='环境温度',
        line=dict(color='#E06C75', width=2),
        marker=dict(size=5, color='#E06C75'),
        fill='tozeroy',
        fillcolor='rgba(224,108,117,0.12)',
        hovertemplate='日期: %{x}<br>温度: %{y:.1f} ℃<extra></extra>'
    ),
    row=1, col=1
)

# 标注最高温
t_max = daily.loc[daily['Ambient_Temperature'].idxmax()]
t_min = daily.loc[daily['Ambient_Temperature'].idxmin()]
fig.add_annotation(
    x=t_max['date_str'], y=t_max['Ambient_Temperature'],
    text=f'最高 {t_max["Ambient_Temperature"]:.1f}℃',
    showarrow=True, arrowhead=1, arrowcolor='#C44E52',
    font=dict(color='#C44E52', size=11), yshift=10,
    row=1, col=1
)
fig.add_annotation(
    x=t_min['date_str'], y=t_min['Ambient_Temperature'],
    text=f'最低 {t_min["Ambient_Temperature"]:.1f}℃',
    showarrow=True, arrowhead=1, arrowcolor='#4C72B0',
    font=dict(color='#4C72B0', size=11), yshift=-15,
    row=1, col=1
)

# 下图：湿度
fig.add_trace(
    go.Scatter(
        x=daily['date_str'], y=daily['Humidity'],
        mode='lines+markers',
        name='空气湿度',
        line=dict(color='#61AFEF', width=2),
        marker=dict(size=5, color='#61AFEF'),
        fill='tozeroy',
        fillcolor='rgba(97,175,239,0.12)',
        hovertemplate='日期: %{x}<br>湿度: %{y:.1f} %<extra></extra>'
    ),
    row=2, col=1
)

# 标注最高/最低湿度
h_max = daily.loc[daily['Humidity'].idxmax()]
h_min = daily.loc[daily['Humidity'].idxmin()]
fig.add_annotation(
    x=h_max['date_str'], y=h_max['Humidity'],
    text=f'最高 {h_max["Humidity"]:.1f}%',
    showarrow=True, arrowhead=1, arrowcolor='#2878A0',
    font=dict(color='#2878A0', size=11), yshift=10,
    row=2, col=1
)
fig.add_annotation(
    x=h_min['date_str'], y=h_min['Humidity'],
    text=f'最低 {h_min["Humidity"]:.1f}%',
    showarrow=True, arrowhead=1, arrowcolor='#E5A93D',
    font=dict(color='#E5A93D', size=11), yshift=-15,
    row=2, col=1
)

# 布局
fig.update_layout(
    title=dict(
        text='茶园环境温度与空气湿度日均变化趋势（交互式）',
        font=dict(size=16), x=0.5
    ),
    showlegend=False,
    hovermode='x unified',
    height=700,
    template='plotly_white'
)
fig.update_xaxes(title_text='日期', row=2, col=1)
fig.update_yaxes(title_text='环境温度 (℃)', row=1, col=1)
fig.update_yaxes(title_text='空气湿度 (%)', row=2, col=1)

# 保存为 HTML（交互式）
fig.write_html('../outputs/figures/fig08_interactive.html')
print("图8（交互式）保存为 HTML 完成，在浏览器中打开可交互缩放、悬停查看数值")

# 同时保存静态 PNG 作为后备
fig.write_image('../outputs/figures/fig08_interactive.png', width=1400, height=700, scale=2)
print("图8 静态备份保存完成")

fig.show()

### 图9：日均温度异常值自动标注

In [ ]:
daily_temp = df.groupby('date')['Ambient_Temperature'].mean().reset_index()
daily_temp.columns = ['date', 'temp']
daily_temp['date_idx'] = range(len(daily_temp))

q1, q3 = daily_temp['temp'].quantile(0.25), daily_temp['temp'].quantile(0.75)
iqr = q3 - q1
daily_temp['is_outlier'] = (daily_temp['temp'] < q1-1.5*iqr) | (daily_temp['temp'] > q3+1.5*iqr)
outlier = daily_temp[daily_temp['is_outlier']]

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(daily_temp['date_idx'], daily_temp['temp'], color='#3498DB', linewidth=2, alpha=0.7, label='日均温度')
ax.fill_between(daily_temp['date_idx'], daily_temp['temp'], alpha=0.1, color='#3498DB')

if len(outlier) > 0:
    ax.scatter(outlier['date_idx'], outlier['temp'], color='#E74C3C', s=120, zorder=5,
               label=f'异常值 ({len(outlier)}个)', edgecolors='darkred', linewidths=1.5)
    for _, row in outlier.iterrows():
        ax.annotate(f'{row["temp"]:.1f}℃', xy=(row['date_idx'], row['temp']),
                    xytext=(row['date_idx']+0.5, row['temp']+0.3),
                    fontsize=9, color='#E74C3C', fontweight='bold',
                    arrowprops=dict(arrowstyle='->', color='#E74C3C', lw=1.2))

max_idx = daily_temp['temp'].idxmax()
min_idx = daily_temp['temp'].idxmin()
ax.annotate(f'最高 {daily_temp.loc[max_idx,"temp"]:.1f}℃',
            xy=(daily_temp.loc[max_idx,'date_idx'], daily_temp.loc[max_idx,'temp']),
            xytext=(daily_temp.loc[max_idx,'date_idx']-2, daily_temp.loc[max_idx,'temp']+0.5),
            fontsize=9, color='darkgreen', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='darkgreen'))
ax.annotate(f'最低 {daily_temp.loc[min_idx,"temp"]:.1f}℃',
            xy=(daily_temp.loc[min_idx,'date_idx'], daily_temp.loc[min_idx,'temp']),
            xytext=(daily_temp.loc[min_idx,'date_idx']+1, daily_temp.loc[min_idx,'temp']-0.8),
            fontsize=9, color='navy', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='navy'))

ax.axhline(q1-1.5*iqr, color='orange', linestyle='--', alpha=0.6, linewidth=1, label=f'IQR下界 ({q1-1.5*iqr:.1f}℃)')
ax.axhline(q3+1.5*iqr, color='orange', linestyle='--', alpha=0.6, linewidth=1, label=f'IQR上界 ({q3+1.5*iqr:.1f}℃)')

ax.set_title('日均环境温度时序分析（含异常值自动标注）', fontsize=14, fontweight='bold')
ax.set_xlabel('监测天数', fontsize=11); ax.set_ylabel('环境温度 (℃)', fontsize=11)
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/figures/fig09_outlier_detection.png', dpi=150, bbox_inches='tight')
plt.show()
print("图9保存完成")


## 4. 统计检验、聚类与回归分析

### 图10：各健康状态下核心指标均值对比（含ANOVA显著性标注）

In [ ]:
from scipy import stats
import numpy as np

# 拆两组：上排(温/湿相关，量级相近) + 下排(光照/营养，光照独大)
top_cols = ['Ambient_Temperature', 'Soil_Moisture', 'Humidity']
top_labels = ['环境温度\n(℃)', '土壤湿度\n(%)', '空气湿度\n(%)']

bottom_cols = ['Light_Intensity', 'Nitrogen_Level', 'Chlorophyll_Content']
bottom_labels = ['光照强度\n(lux)', '氮含量\n(mg/kg)', '叶绿素\n(SPAD)']

all_cols = top_cols + bottom_cols
all_labels = top_labels + bottom_labels

# 计算三组均值 + ANOVA
means, f_vals, p_vals = {}, [], []
for col in all_cols:
    grp = df_clean.groupby('健康状态')[col].mean()
    means[col] = {'健康': grp.get('健康',0), '中度胁迫': grp.get('中度胁迫',0), '高度胁迫': grp.get('高度胁迫',0)}
    groups = [df_clean[df_clean['健康状态']==s][col].dropna() for s in ['健康','中度胁迫','高度胁迫']]
    f, p = stats.f_oneway(*groups)
    f_vals.append(f); p_vals.append(p)

def sig_label(p):
    if p < 0.001: return '极显著'
    if p < 0.01: return '很显著'
    if p < 0.05: return '显著'
    return '不显著'

results_df = pd.DataFrame({
    '指标': [l.replace('\n','') for l in all_labels],
    'F值': [round(f,2) for f in f_vals],
    'p值': [f'{p:.4f}' if p>=0.0001 else '<0.0001' for p in p_vals],
    '显著性': [sig_label(p) for p in p_vals]
})

# ===== 上下两个子图，各有独立Y轴，光照不再碾压其他 =====
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 12))
fig.suptitle('不同健康状态下各环境与营养指标均值对比', fontsize=16, fontweight='bold')

width = 0.25
status_order = ['健康', '中度胁迫', '高度胁迫']
bar_colors = {'健康': '#59A96C', '中度胁迫': '#E5A93D', '高度胁迫': '#E0555A'}

for ax, cols, labels in [(ax1, top_cols, top_labels),
                          (ax2, bottom_cols, bottom_labels)]:
    x = np.arange(len(labels))

    for j, status in enumerate(status_order):
        offset = (j - 1) * width
        bars = ax.bar(x + offset, [means[c][status] for c in cols],
                      width, color=bar_colors[status], label=status, edgecolor='white')
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.5,
                    f'{h:.1f}', ha='center', va='bottom', fontsize=8)

    # 显著性标注 —— 每个指标用自己的柱高
    for i, col in enumerate(cols):
        p = p_vals[all_cols.index(col)]
        y_pos = max(means[col].values()) * 1.12
        label = sig_label(p)
        c = '#E0555A' if p < 0.05 else '#9AACB8'
        fw = 'bold' if p < 0.05 else 'normal'
        ax.text(i, y_pos, label, ha='center', fontsize=12, color=c, fontweight=fw)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=11)
    ax.set_ylabel('均值', fontsize=12)
    ax.legend(fontsize=10, loc='upper right')
    ax.grid(axis='y', alpha=0.3)

# 底部说明
ax2.text(0.5, -0.15,
         '柱顶标注：极显著 = p<0.001 | 很显著 = p<0.01 | 显著 = p<0.05 | 不显著 = 无统计学差异',
         transform=ax2.transAxes, fontsize=9, ha='center', color='gray')

plt.tight_layout()
plt.savefig('../outputs/figures/fig10_anova.png', dpi=150, bbox_inches='tight')
plt.show()
print("图10保存完成")
print(results_df.to_string(index=False))

### 图11：K-Means聚类分析

In [ ]:

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

cluster_cols = ['Ambient_Temperature','Humidity','Soil_Moisture','Light_Intensity','Nitrogen_Level','Chlorophyll_Content']
X = df_clean[cluster_cols].dropna()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('植株环境指标K-Means聚类分析', fontsize=14, fontweight='bold')

inertias = []
K_range = range(2, 8)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].axvline(x=3, color='red', linestyle='--', linewidth=1.5, label='选定K=3')
axes[0].set_xlabel('聚类数量K', fontsize=11); axes[0].set_ylabel('簇内误差平方和', fontsize=11)
axes[0].set_title('肘部法则确定最优K值', fontsize=12)
axes[0].legend(); axes[0].grid(alpha=0.3)

km3 = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = km3.fit_predict(X_scaled)
df_cluster = df_clean.loc[X.index].copy()
df_cluster['聚类'] = [f'群组{l+1}' for l in labels]

cluster_colors = {'群组1': '#E74C3C', '群组2': '#3498DB', '群组3': '#27AE60'}
for cluster, group in df_cluster.groupby('聚类'):
    axes[1].scatter(group['Ambient_Temperature'], group['Chlorophyll_Content'],
                    c=cluster_colors[cluster], label=cluster, alpha=0.6, s=30, edgecolors='none')

axes[1].set_xlabel('环境温度 (℃)', fontsize=11); axes[1].set_ylabel('叶绿素含量 (SPAD)', fontsize=11)
axes[1].set_title('K-Means聚类结果（温度 vs 叶绿素）', fontsize=12)
axes[1].legend(title='聚类群组'); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/fig11_kmeans.png', dpi=150, bbox_inches='tight')
plt.show()
print("图11保存完成")


### 图12：光照强度对叶绿素含量线性回归

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

X_reg = df_clean[['Light_Intensity']].values
y_reg = df_clean['Chlorophyll_Content'].values
reg = LinearRegression()
reg.fit(X_reg, y_reg)
y_pred = reg.predict(X_reg)
r2 = r2_score(y_reg, y_pred)
residuals = y_reg - y_pred

x_line = np.linspace(X_reg.min(), X_reg.max(), 100).reshape(-1,1)
y_line = reg.predict(x_line)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('光照强度对叶绿素含量线性回归分析', fontsize=14, fontweight='bold')

axes[0].scatter(X_reg, y_reg, alpha=0.3, s=15, color='#3498DB', edgecolors='none', label='观测值')
axes[0].plot(x_line, y_line, color='#E74C3C', linewidth=2.5,
             label=f'回归线: y={reg.coef_[0]:.4f}x+{reg.intercept_:.2f}')
ci = 1.96 * np.std(residuals)
axes[0].fill_between(x_line.flatten(), y_line-ci, y_line+ci, alpha=0.15, color='#E74C3C', label='95%置信区间')
axes[0].set_xlabel('光照强度 (lux)', fontsize=11); axes[0].set_ylabel('叶绿素含量 (SPAD)', fontsize=11)
axes[0].set_title(f'线性回归 (R$^2$={r2:.4f})', fontsize=12)
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

axes[1].scatter(y_pred, residuals, alpha=0.3, s=15, color='#9B59B6', edgecolors='none')
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('预测值', fontsize=11); axes[1].set_ylabel('残差', fontsize=11)
axes[1].set_title('残差分布图', fontsize=12); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/fig12_regression.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"图12保存完成 | R²={r2:.4f} | 回归方程: y={reg.coef_[0]:.4f}x+{reg.intercept_:.2f}")

### 图13：电化学信号与健康状态关联分析

In [ ]:
# ===== 图13：电化学信号与健康状态关联分析 =====
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('电化学信号与植物健康状态关联分析', fontsize=14, fontweight='bold')

# 左图：小提琴图（按健康状态）
order = ['健康', '中度胁迫', '高度胁迫']
violin_parts = axes[0].violinplot(
    [df_clean[df_clean['健康状态']==s]['Electrochemical_Signal'].dropna() for s in order],
    positions=[0, 1, 2],
    showmeans=True, showmedians=True
)
for i, pc in enumerate(violin_parts['bodies']):
    pc.set_facecolor(list(status_palette.values())[i])
    pc.set_alpha(0.7)
axes[0].set_xticks([0, 1, 2])
axes[0].set_xticklabels(order, fontsize=11)
axes[0].set_ylabel('电化学信号', fontsize=11)
axes[0].set_title('不同健康状态下电化学信号分布', fontsize=12)
axes[0].grid(axis='y', alpha=0.3)

# 右图：电化学信号 vs 土壤湿度散点（颜色=健康状态）
for status, color in status_palette.items():
    subset = df_clean[df_clean['健康状态'] == status]
    axes[1].scatter(subset['Soil_Moisture'], subset['Electrochemical_Signal'],
                    c=color, label=status, alpha=0.4, s=20, edgecolors='none')
axes[1].set_xlabel('土壤湿度 (%)', fontsize=11)
axes[1].set_ylabel('电化学信号', fontsize=11)
axes[1].set_title('电化学信号 vs 土壤湿度', fontsize=12)
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

# 统计摘要
print("电化学信号统计摘要（按健康状态）：")
print(df_clean.groupby('健康状态')['Electrochemical_Signal'].describe().round(3))

plt.tight_layout()
plt.savefig('../outputs/figures/fig13_electrochemical.png', dpi=150, bbox_inches='tight')
plt.show()
print("图13 保存完成")

## 5. AI智能结论分析（DeepSeek API）

In [ ]:
import os
from openai import OpenAI

# ===== DeepSeek AI 分析（需设置 DEEPSEEK_API_KEY 环境变量）=====
api_key = os.environ.get("DEEPSEEK_API_KEY")
if api_key:
    client = OpenAI(
        api_key=api_key,
        base_url="https://api.deepseek.com",
        timeout=60.0,
        max_retries=2,
    )
else:
    client = None
    print("⚠️ 未设置 DEEPSEEK_API_KEY 环境变量，跳过 AI 分析环节")
    print("   Windows: set DEEPSEEK_API_KEY=sk-xxx")
    print("   macOS/Linux: export DEEPSEEK_API_KEY=sk-xxx")

if client is not None:
    # 整理分析数据摘要
    summary = f"""
数据集：茶园植物健康环境监测数据，1200条记录，10株植物，2024年10月
健康状态分布：{df_clean['Plant_Health_Status'].value_counts().to_dict()}

各健康状态平均指标：
{df_clean.groupby('Plant_Health_Status')[['Ambient_Temperature','Soil_Moisture','Humidity','Chlorophyll_Content','Nitrogen_Level']].mean().round(2).to_string()}

ANOVA检验结果：
{results_df[['指标','F值','p值','显著性']].to_string(index=False)}

线性回归：光照强度 → 叶绿素含量
回归方程: y = {reg.coef_[0]:.4f}x + {reg.intercept_:.2f}，R² = {r2:.4f}

K-Means聚类：K=3，按环境温度、湿度、土壤湿度、光照、氮含量、叶绿素分群
"""

    # 第一轮：生成数据分析结论
    print("【第一轮】生成数据分析结论...")
    messages = [
        {"role": "system", "content": "你是一位专业的农业数据分析专家，擅长植物生理与环境监测数据解读，请用中文给出专业、简洁的分析结论。"},
        {"role": "user", "content": f"请根据以下茶园环境监测数据分析结果，生成4-6条专业的数据分析结论，每条结论需结合具体数据数字：\n{summary}"}
    ]
    try:
        response1 = client.chat.completions.create(
            model="deepseek-chat", messages=messages, max_tokens=800
        )
        conclusion = response1.choices[0].message.content
        print("\nDeepSeek AI 数据分析结论：")
        print(conclusion)

        # 第二轮：追问茶园管理建议
        print("\n【第二轮】生成茶园管理建议...")
        messages.append({"role": "assistant", "content": conclusion})
        messages.append({"role": "user", "content": "基于以上分析结论，请给出3-5条具体可操作的茶园精准管理建议，包括灌溉、光照控制、营养管理等方面。"})
        response2 = client.chat.completions.create(
            model="deepseek-chat", messages=messages, max_tokens=600
        )
        suggestions = response2.choices[0].message.content
        print("\nDeepSeek AI 茶园管理建议：")
        print(suggestions)

        # 第三轮：生成报告摘要
        print("\n【第三轮】生成项目报告摘要...")
        messages.append({"role": "assistant", "content": suggestions})
        messages.append({"role": "user", "content": "请将以上分析结论和管理建议整合成一段100-150字的项目报告摘要，语言简洁专业。"})
        response3 = client.chat.completions.create(
            model="deepseek-chat", messages=messages, max_tokens=300
        )
        abstract = response3.choices[0].message.content
        print("\nDeepSeek AI 项目报告摘要：")
        print(abstract)
    except Exception as e:
        print(f"\n⚠️ AI 分析出错: {e}")
        print("请检查网络连接或 API Key 是否有效")
else:
    print("\n（AI 分析环节已跳过——未配置 DEEPSEEK_API_KEY）")

## 6. 综合分析总结

### 核心发现

| 序号 | 发现 | 数据支撑 |
|------|------|----------|
| 1 | **土壤湿度是第一驱动因子** | ANOVA F=899.26，p<0.0001；健康组34.91% vs 高胁迫18.08% |
| 2 | **氮含量显著影响健康** | ANOVA F=46.87，p<0.0001；健康组34.79 vs 高胁迫26.97 mg/kg |
| 3 | **光照→叶绿素几乎无线性关系** | R²=0.0003，回归系数仅0.0007 |
| 4 | **茶园整体健康堪忧** | 健康仅24.9%，中高胁迫合计75.1% |
| 5 | **温湿度非健康区分因子** | ANOVA p>0.05，三组间均值差异极小 |

### 管理建议优先级

1. 🚿 **立即行动** — 智能分区灌溉，目标土壤湿度≥35%
2. 🧪 **短期跟进** — 高胁迫区氮肥梯度追施，目标≥30 mg/kg
3. 📊 **中期优化** — 基于K-Means聚类实施差异化管理
4. 🔬 **持续监测** — 用SPAD叶绿素仪跟踪水肥调控效果
5. 💰 **资源优化** — 暂缓温湿度/光照硬件投入，转向土壤养分检测

## 7. 项目说明

**数据来源**：Kaggle公开数据集 plant_health_data.csv，真实生物传感器采集，1200条记录，10株植物，时间范围2024年10月。

**数据字段**：环境温度、土壤温度、空气湿度、土壤湿度、光照强度、土壤pH、氮/磷/钾含量、叶绿素含量、植物健康状态。

**数据处理**：IQR方法处理异常值；衍生气土温差与综合营养指数两个指标。

**运行方式**：
```
pip install -r requirements.txt
jupyter notebook notebooks/analysis.ipynb
```

**局限性**：数据时间范围仅约一个月，植株样本10株，结论需结合更长周期数据验证。
